In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from datetime import datetime


In [2]:
dataset = "impli"

sim_functions = [
    "cos",
    "cka",
    "wasser",
]

agg_functions = [
    "mean",
    "max",
    "sum",
    "entropy",
    "gini"
]


model_map = {
    "google-bert/bert-base-uncased": "BERT-base Uncased",
    "google-bert/bert-base-cased": "BERT-base Cased",
    "google-bert/bert-large-uncased": "BERT-large Uncased",
    "google-bert/bert-large-cased": "BERT-large Cased",
    "answerdotai/ModernBERT-base": "ModernBERT-base",
    "answerdotai/ModernBERT-large": "ModernBERT-large",
}

models = list(model_map.keys())

layer_map = {
    "google-bert/bert-base-uncased": list(range(12)),
    "google-bert/bert-base-cased": list(range(12)),
    "google-bert/bert-large-uncased": list(range(24)),
    "google-bert/bert-large-cased": list(range(24)),
    "answerdotai/ModernBERT-base": list(range(22)),
    "answerdotai/ModernBERT-large": list(range(28)),
}

# scores_dir
ROOT = Path.cwd().parent

print(ROOT)

scores_dir = ROOT / "decomp_measure" / "scores" / "impli_layers"

print("Notebook cwd:", Path.cwd())
print("Scores dir:", scores_dir)
print("Exists:", scores_dir.exists())

def model_to_folder(model_name: str) -> str:
    return model_name.replace("/", "_")


/Users/mmi/Documents/projects/idioms_decomposability/decomp_code/idioms_decomposability
Notebook cwd: /Users/mmi/Documents/projects/idioms_decomposability/decomp_code/idioms_decomposability/correlation_experiment
Scores dir: /Users/mmi/Documents/projects/idioms_decomposability/decomp_code/idioms_decomposability/decomp_measure/scores/impli_layers
Exists: True


In [3]:
# regex for filename like: impli_cka_entropy_bert-large-cased.csv
fname_re = re.compile(r"^(?P<dataset>[^_]+)_(?P<sim>[^_]+)_(?P<agg>[^_]+)_(?P<modeltag>.+)\.csv$")

records = []

for model_name in models:
    model_folder = model_to_folder(model_name)
    allowed_layers = set(layer_map[model_name])

    model_root = scores_dir / model_folder
    if not model_root.exists():
        print(f"[skip] missing model folder: {model_root}")
        continue

    # timestamps are subfolders; search all of them
    for csv_path in model_root.rglob(f"{dataset}_*.csv"):
        # parse layer from .../layer_X/...
        try:
            layer_part = next(p for p in csv_path.parts if p.startswith("layer_"))
            layer = int(layer_part.split("_")[1])
        except StopIteration:
            continue
        except ValueError:
            continue

        if layer not in allowed_layers:
            continue

        m = fname_re.match(csv_path.name)
        if not m:
            continue

        ds = m.group("dataset")
        sim = m.group("sim")
        agg = m.group("agg")
        if ds != dataset or sim not in sim_functions or agg not in agg_functions:
            continue

        # timestamp is the folder just under model folder
        # .../impli_layers/<model_folder>/<timestamp>/layer_X/...
        try:
            timestamp = csv_path.relative_to(model_root).parts[0]
        except Exception:
            timestamp = None

        df = pd.read_csv(csv_path)
        df["model_name"] = model_name
        df["model_label"] = model_map[model_name]
        df["model_folder"] = model_folder
        df["timestamp"] = timestamp
        df["layer"] = layer
        df["sim_func"] = sim
        df["agg_func"] = agg
        df["source_path"] = str(csv_path)

        records.append(df)

if not records:
    # helpful debug so you don't get the pandas concat error
    any_csv = list(scores_dir.rglob("*.csv"))[:10]
    print("No matching CSVs loaded.")
    print("scores_dir:", scores_dir)
    print("scores_dir exists?", scores_dir.exists())
    print("Example CSVs under scores_dir:", [str(p) for p in any_csv])
else:
    all_scores = pd.concat(records, ignore_index=True)
    print(f"Loaded {len(all_scores):,} rows from {len(records):,} files")
    all_scores.head()

Loaded 964,410 rows from 1,830 files


In [4]:
all_scores = pd.concat(records, ignore_index=True)
all_scores.shape

all_scores.head()

,Unnamed: 0,premise,hypothesis,extracted_idiom,base_form,idiom_pos,idiom_processed,token_scores,decomp_score,sim_function,model_name,model_label,model_folder,timestamp,layer,sim_func,agg_func,source_path
0,0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,"[weather, the, storm]",weathered the storm,"[(4, 'weathered', -0.0275840163230896), (5, 't...",0.027584,cos,google-bert/bert-base-uncased,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...
1,1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,"[against, the, grain]",against the grain,"[(25, 'against', 0.015303194522857666), (26, '...",0.015303,cos,google-bert/bert-base-uncased,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...
2,2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,"[go, it, alone]",going it alone,"[(8, 'going', 0.004256725311279297), (9, 'it',...",0.023199,cos,google-bert/bert-base-uncased,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...
3,3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,"[come, clean]",come clean,"[(19, 'come', 0.0060558319091796875), (20, 'cl...",0.006056,cos,google-bert/bert-base-uncased,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...
4,4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,"[rule, the, roost]",rule the roost,"[(27, 'rule', 0.00505441427230835), (28, 'the'...",0.006530,cos,google-bert/bert-base-uncased,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...


In [5]:
all_scores.to_csv(ROOT / "data" / "processed" / "impli_layers.csv", index=False)

# Get correlation

In [ ]:
df = pd.read_csv(ROOT / "data" / "processed" / "impli_layers.csv")

print(df.shape)




(964410, 18)
